# 06. Build the IVF index: cluster the vectors

Groups all review vectors into 50 clusters with k-means and saves the cluster centres.

## What to look at

- The **centroids table**: 50 rows, with how many reviews each cluster owns.
- The **bar chart**: cluster sizes (some clusters are much bigger, e.g. generic one-word praise).
- This is *our* technique: Iceberg has no vector index.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
run_script("06_build_ivf_index.py")

In [ ]:
spark = get_spark("nb06")
cent = sql(spark, f"SELECT cluster_id, size, slice(centroid, 1, 3) AS first_3_numbers, size(centroid) AS dim FROM {T_CENTROIDS} ORDER BY cluster_id")
cent.head(10)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(cent.cluster_id, cent["size"], color="#2f6fed")
ax.set_xlabel("cluster_id"); ax.set_ylabel("reviews in cluster"); ax.set_title(f"{len(cent)} clusters")
plt.show()

In [ ]:
print("clusters:", len(cent), "| reviews:", int(cent["size"].sum()), "| smallest/median/largest:", int(cent["size"].min()), int(cent["size"].median()), int(cent["size"].max()))

In [ ]:
# Cleanup. Skip this cell while you are showing the Spark UI (http://localhost:4040): it closes with the session.
stop_spark()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

New folder `gold/centroids/`: a tiny table (a handful of very small Parquet files under **data/**, plus **metadata/**). It is separate from the big gold table, which comes next.